# Studi Kasus 3: Evaluasi Kurikulum Berbasis Tracer Study Alumni (Mode Lokal)

Notebook ini merupakan versi rilis publik **100% lokal & teranonimisasi** untuk pilar **Evaluasi Kurikulum & Outcome Karir Alumni**, membaca langsung dari `../data/curriculum_tracer_anonymized.csv`.

**Fokus Analitik (DS / ML / DA / BI):**
1. Profil nilai modul praktik & waktu tunggu kerja untuk kelompok **Bekerja**, **Buka Usaha**, dan **Belum Bekerja**.
2. Analisis korelasi statistik (Pearson $r$) modul terhadap serapan karir dan kecepatan mendapat kerja.
3. Replikasi `DecisionTreeClassifier` Skema A (4 modul) dan Skema B (4 modul + Skor Uji Kompetensi).
4. Uji inferensi dengan simulasi *missing value* (`None`) melalui `src/inference.py`.

In [1]:
import sys
from pathlib import Path
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data_loader import load_curriculum_tracer_data
from src.inference import VocationalInferenceEngine

df = load_curriculum_tracer_data()
serapan = (df['status_bekerja'] != 'Belum Bekerja').mean() * 100
print(f"Dataset alumni lokal: {len(df)} alumni valid | Total Tingkat Serapan: {serapan:.2f}%")
print(df.head(5).to_string(index=False))

Dataset alumni lokal: 315 alumni valid | Total Tingkat Serapan: 77.14%
alumni_id  nilai_ukur_pola  nilai_jahit_dasar  nilai_jahit_lanjut  nilai_finishing  skor_uji_kompetensi status_bekerja  waktu_tunggu_bulan  status_encoded
  ALM-001             90.4               82.8                90.6             87.2                 88.6     Buka Usaha                   4               1
  ALM-002             86.0               90.2                79.9             89.1                 86.6        Bekerja                   4               0
  ALM-003             71.2               83.6                75.5             86.0                 79.8        Bekerja                   5               0
  ALM-004             88.8               77.9                86.6             81.6                 83.9     Buka Usaha                   7               1
  ALM-005             75.0               84.1                76.7             90.6                 83.2        Bekerja                   4               0

## 1. Profil Kompetensi & Korelasi Statistik per Jalur Karir

In [2]:
modul_cols = ['nilai_ukur_pola', 'nilai_jahit_dasar', 'nilai_jahit_lanjut', 'nilai_finishing', 'skor_uji_kompetensi']
print("=== Rata-Rata Nilai Modul & Waktu Tunggu per Status Alumni ===")
print(df.groupby('status_bekerja')[modul_cols + ['waktu_tunggu_bulan']].mean().round(2).to_string())

df['terserap_biner'] = (df['status_bekerja'] != 'Belum Bekerja').astype(int)
print("\n=== Korelasi Pearson (r) terhadap Serapan Kerja & Waktu Tunggu ===")
corr_rows = []
df_emp = df[df['terserap_biner'] == 1]
for col in modul_cols:
    r_serap = df[col].corr(df['terserap_biner'])
    r_tunggu = df_emp[col].corr(df_emp['waktu_tunggu_bulan'])
    corr_rows.append({'komponen': col, 'r_terserap': round(r_serap, 3), 'r_waktu_tunggu': round(r_tunggu, 3)})
print(pd.DataFrame(corr_rows).to_string(index=False))

=== Rata-Rata Nilai Modul & Waktu Tunggu per Status Alumni ===
                nilai_ukur_pola  nilai_jahit_dasar  nilai_jahit_lanjut  nilai_finishing  skor_uji_kompetensi  waktu_tunggu_bulan
status_bekerja                                                                                                                  
Bekerja                   80.41              87.96               81.89            87.17                85.42                4.54
Belum Bekerja             71.42              73.85               70.20            70.15                71.07               18.15
Buka Usaha                89.09              83.06               88.31            81.55                86.77                5.49

=== Korelasi Pearson (r) terhadap Serapan Kerja & Waktu Tunggu ===
           komponen  r_terserap  r_waktu_tunggu
    nilai_ukur_pola       0.681           0.031
  nilai_jahit_dasar       0.752          -0.387
 nilai_jahit_lanjut       0.756           0.013
    nilai_finishing       0.800 

## 2. Replikasi Evaluasi Decision Tree (Skema A & Skema B)

In [3]:
feat_a = ['nilai_ukur_pola', 'nilai_jahit_dasar', 'nilai_jahit_lanjut', 'nilai_finishing']
feat_b = feat_a + ['skor_uji_kompetensi']
y = df['status_encoded']

idx_tr, idx_te = train_test_split(df.index, test_size=0.2, random_state=42, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe_a = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('classifier', DecisionTreeClassifier(criterion='gini', max_depth=5, min_samples_leaf=3, min_samples_split=4, random_state=42))
])
pipe_b = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('classifier', DecisionTreeClassifier(criterion='entropy', max_depth=5, min_samples_leaf=3, min_samples_split=10, random_state=42))
])

for name, pipe, feats in [('Skema A (4 Modul)', pipe_a, feat_a), ('Skema B (4 Modul + Uji Komp)', pipe_b, feat_b)]:
    cv_f1 = cross_val_score(pipe, df.loc[idx_tr, feats], y.loc[idx_tr], cv=cv, scoring='f1_macro').mean()
    pipe.fit(df.loc[idx_tr, feats], y.loc[idx_tr])
    preds = pipe.predict(df.loc[idx_te, feats])
    print(f"{name:30s} -> CV Macro-F1: {cv_f1:.4f} | Test Acc: {accuracy_score(y.loc[idx_te], preds):.4f} | Test Macro-F1: {f1_score(y.loc[idx_te], preds, average='macro'):.4f}")

Skema A (4 Modul)              -> CV Macro-F1: 0.8951 | Test Acc: 0.9365 | Test Macro-F1: 0.9437
Skema B (4 Modul + Uji Komp)   -> CV Macro-F1: 0.9286 | Test Acc: 0.9048 | Test Macro-F1: 0.9155


## 3. Uji Inferensi Artefak Model Ter-deploy (Termasuk Penanganan Missing Value `None`)

In [4]:
engine = VocationalInferenceEngine()
res = engine.predict_curriculum_outcome(
    nilai_ukur_pola=87.5,
    nilai_jahit_dasar=82.0,
    nilai_jahit_lanjut=None,  # Uji imputer internal pipeline
    nilai_finishing=81.5,
    skor_uji_kompetensi=85.0
)
for k, v in res.items():
    print(f"{k}: {v}")

skema_a_4modul: {'code': 1, 'label': 'Buka Usaha (Wirausaha Mandiri)', 'probabilities': {'Bekerja': 0.0, 'Buka Usaha': 1.0, 'Belum Bekerja': 0.0}}
skema_b_komprehensif: {'code': 1, 'label': 'Buka Usaha (Wirausaha Mandiri)', 'probabilities': {'Bekerja': 0.0, 'Buka Usaha': 1.0, 'Belum Bekerja': 0.0}}
recommendation: Berikan pendampingan bisnis mikro, penetapan harga jasa, dan pemasaran digital untuk usaha mandiri.
